# 📊 Notebook 01 — Analyse Exploratoire des Données (EDA)

## Objectif
L'**EDA (Exploratory Data Analysis)** est la première étape de tout projet ML.  
Elle permet de **comprendre les données** avant de construire un modèle.

### Ce qu'on va faire :
1. Vue d'ensemble du dataset
2. Analyse de la variable cible `SalePrice`
3. Test de normalité et transformation logarithmique
4. Analyse des corrélations
5. Détection des outliers
6. Analyse des valeurs manquantes
7. Analyse des variables catégorielles
8. Identification des relations non-linéaires

> **Dataset** : House Prices - Ames, Iowa (USA) — 1460 maisons, 79 variables


In [ ]:
# ============================================================
# CELLULE 1 : Importation des bibliothèques
# ============================================================
# pandas : manipulation de tableaux de données (comme Excel mais en Python)
import pandas as pd

# numpy : calculs mathématiques (tableaux, statistiques)
import numpy as np

# matplotlib & seaborn : création de graphiques
import matplotlib.pyplot as plt
import seaborn as sns

# scipy.stats : tests statistiques (test de normalité, etc.)
from scipy import stats

# Paramètres d'affichage
pd.set_option('display.max_columns', 50)   # Affiche jusqu'à 50 colonnes
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 décimales

# Style graphique
plt.style.use('seaborn-v0_8-whitegrid')  # Fond blanc avec grille
sns.set_palette("husl")                   # Palette de couleurs

print("✅ Bibliothèques importées avec succès")


In [ ]:
# ============================================================
# CELLULE 2 : Chargement des données
# ============================================================
# pd.read_csv() lit un fichier CSV (valeurs séparées par des virgules)
# et le transforme en DataFrame (tableau 2D avec lignes et colonnes)
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f"📊 Dataset d'entraînement : {train.shape[0]} lignes × {train.shape[1]} colonnes")
print(f"📊 Dataset de test        : {test.shape[0]} lignes × {test.shape[1]} colonnes")
print()
print("Aperçu des premières lignes du train :")
train.head()


In [ ]:
# ============================================================
# CELLULE 3 : Types de données et statistiques de base
# ============================================================
print("=== Types de colonnes ===")
# On compte le nombre de colonnes de chaque type
type_counts = train.dtypes.value_counts()
print(type_counts)
print()

# dtypes retourne le type de chaque colonne
# object = texte (variables catégorielles)
# int64/float64 = nombres
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
categ_cols   = train.select_dtypes(include=['object']).columns.tolist()

print(f"Variables numériques  : {len(numeric_cols)}")
print(f"Variables catégorielles : {len(categ_cols)}")
print()

# describe() calcule les statistiques descriptives de base
print("=== Statistiques descriptives (premières colonnes numériques) ===")
train[['SalePrice', 'GrLivArea', 'LotArea', 'YearBuilt', 'OverallQual']].describe()


## 1. Analyse de la Variable Cible : `SalePrice`

La variable cible est ce qu'on cherche à prédire : le **prix de vente** (`SalePrice`).

Avant tout, on doit comprendre sa **distribution** :
- Est-elle symétrique ?  
- Y a-t-il des valeurs extrêmes ?
- Suit-elle une loi normale ?

Cette analyse va justifier (ou non) une **transformation logarithmique**.


In [ ]:
# ============================================================
# CELLULE 4 : Distribution de SalePrice
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution de SalePrice (Variable Cible)', fontsize=14, fontweight='bold')

# --- Histogramme ---
# Un histogramme divise les valeurs en intervalles et compte les occurrences
axes[0].hist(train['SalePrice'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Histogramme de SalePrice', fontsize=12)
axes[0].set_xlabel('Prix de Vente ($)')
axes[0].set_ylabel('Nombre de maisons')
# Ligne verticale pour la moyenne
axes[0].axvline(train['SalePrice'].mean(), color='red', linestyle='--',
                linewidth=2, label=f'Moyenne: ${train["SalePrice"].mean():,.0f}')
axes[0].axvline(train['SalePrice'].median(), color='green', linestyle='--',
                linewidth=2, label=f'Médiane: ${train["SalePrice"].median():,.0f}')
axes[0].legend()

# --- Boxplot ---
# Le boxplot montre : médiane, Q1, Q3, et les outliers (points au-delà des moustaches)
axes[1].boxplot(train['SalePrice'], patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='navy'))
axes[1].set_title('Boxplot de SalePrice', fontsize=12)
axes[1].set_ylabel('Prix de Vente ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# --- Q-Q Plot (Quantile-Quantile) ---
# Compare la distribution observée à une distribution normale théorique
# Si les points sont sur la diagonale → distribution normale
stats.probplot(train['SalePrice'], dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot (Test de Normalité)', fontsize=12)

plt.tight_layout()
plt.savefig('../models/01_saleprice_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistiques
print(f"Moyenne    : ${train['SalePrice'].mean():,.0f}")
print(f"Médiane    : ${train['SalePrice'].median():,.0f}")
print(f"Écart-type : ${train['SalePrice'].std():,.0f}")
print(f"Asymétrie (Skewness) : {train['SalePrice'].skew():.4f}")
print(f"  → Une asymétrie > 1 indique une distribution très asymétrique vers la droite")
print(f"Aplatissement (Kurtosis) : {train['SalePrice'].kurtosis():.4f}")
print(f"  → Un kurtosis élevé = queue lourde (beaucoup d'outliers potentiels)")


In [ ]:
# ============================================================
# CELLULE 5 : Test de normalité de Shapiro-Wilk + Transformation Log
# ============================================================
# Le test de Shapiro-Wilk teste l'hypothèse nulle :
# H0 : la variable suit une distribution normale
# Si p-value < 0.05 → on rejette H0 → la variable N'EST PAS normale

# Test sur SalePrice brut
stat_raw, p_raw = stats.shapiro(train['SalePrice'].sample(min(5000, len(train)),
                                                           random_state=42))
print(f"Test de Shapiro-Wilk sur SalePrice :")
print(f"  Statistique W = {stat_raw:.6f}")
print(f"  p-value = {p_raw:.6e}")
print(f"  → {'NON NORMAL' if p_raw < 0.05 else 'NORMAL'} (p {'<' if p_raw < 0.05 else '>'} 0.05)")

print()

# Transformation logarithmique
# np.log1p(x) = log(x + 1) — le +1 évite log(0) pour les zéros
y_log = np.log1p(train['SalePrice'])

stat_log, p_log = stats.shapiro(y_log.sample(min(5000, len(y_log)), random_state=42))
print(f"Test de Shapiro-Wilk sur log(SalePrice) :")
print(f"  Statistique W = {stat_log:.6f}")
print(f"  p-value = {p_log:.6e}")
print(f"  → {'NON NORMAL' if p_log < 0.05 else 'NORMAL'} (p {'<' if p_log < 0.05 else '>'} 0.05)")

print()
print(f"Asymétrie avant log : {train['SalePrice'].skew():.4f}")
print(f"Asymétrie après log : {y_log.skew():.4f}")
print(f"→ La transformation log RÉDUIT l'asymétrie de {abs(train['SalePrice'].skew()):.2f} à {abs(y_log.skew()):.2f}")


In [ ]:
# ============================================================
# CELLULE 6 : Comparaison SalePrice brut vs log(SalePrice)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparaison : SalePrice vs log(SalePrice)', fontsize=14, fontweight='bold')

y_log = np.log1p(train['SalePrice'])

# SalePrice brut - Histogramme
axes[0, 0].hist(train['SalePrice'], bins=60, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[0, 0].set_title(f'SalePrice Brut (asymétrie = {train["SalePrice"].skew():.2f})', fontsize=11)
axes[0, 0].set_xlabel('Prix ($)')

# SalePrice brut - Q-Q Plot
stats.probplot(train['SalePrice'], plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot - SalePrice Brut', fontsize=11)

# log(SalePrice) - Histogramme
axes[1, 0].hist(y_log, bins=60, color='#27ae60', edgecolor='white', alpha=0.8)
axes[1, 0].set_title(f'log(SalePrice) (asymétrie = {y_log.skew():.2f})', fontsize=11)
axes[1, 0].set_xlabel('log(Prix + 1)')

# log(SalePrice) - Q-Q Plot
stats.probplot(y_log, plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot - log(SalePrice)', fontsize=11)

plt.tight_layout()
plt.savefig('../models/02_log_transform.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 CONCLUSION : log(SalePrice) est plus proche d'une distribution normale.")
print("   → On utilisera log(SalePrice) comme variable cible pour l'entraînement.")
print("   → On fera np.expm1(prediction) pour retrouver le prix en dollars.")


## 2. Analyse des Corrélations

La **corrélation** mesure la relation linéaire entre deux variables.  
Elle varie entre -1 et +1 :
- **+1** : relation positive parfaite (si X augmente, Y augmente)
- **0** : aucune relation linéaire
- **-1** : relation négative parfaite (si X augmente, Y diminue)

Ici, on cherche les variables les plus **corrélées avec SalePrice**.


In [ ]:
# ============================================================
# CELLULE 7 : Matrice de corrélation complète
# ============================================================
# On sélectionne uniquement les colonnes numériques
numeric_train = train.select_dtypes(include=[np.number])

# .corr() calcule le coefficient de Pearson entre toutes les paires de colonnes
corr_matrix = numeric_train.corr()

# Corrélations avec SalePrice, triées par valeur absolue
saleprice_corr = corr_matrix['SalePrice'].abs().sort_values(ascending=False)
print("Top 20 des variables les plus corrélées avec SalePrice :")
print(saleprice_corr.head(20))


In [ ]:
# ============================================================
# CELLULE 8 : Heatmap des 15 variables les plus corrélées
# ============================================================
# On prend les 15 variables les plus corrélées avec SalePrice
top_15 = saleprice_corr.head(16).index.tolist()  # 16 car SalePrice elle-même est incluse

# Sous-matrice de corrélation
corr_top = train[top_15].corr()

fig, ax = plt.subplots(figsize=(14, 12))

# sns.heatmap : matrice colorée
# annot=True : affiche les valeurs dans chaque cellule
# cmap='RdYlGn' : rouge (corrélation négative) → jaune → vert (positive)
# vmin/vmax : plage de couleurs
mask = np.triu(np.ones_like(corr_top, dtype=bool))  # Cache le triangle supérieur (symétrique)
sns.heatmap(corr_top,
            mask=mask,
            annot=True,
            fmt='.2f',
            cmap='RdYlGn',
            vmin=-1, vmax=1,
            center=0,
            ax=ax,
            linewidths=0.5,
            annot_kws={'size': 10})

ax.set_title('Matrice de Corrélation — Top 15 Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/03_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 OBSERVATIONS :")
print("   - OverallQual (qualité générale) : corrélation la plus forte avec SalePrice")
print("   - GrLivArea (surface habitable) : très corrélé aussi")
print("   - Attention : GarageCars et GarageArea sont très corrélés entre eux (multicolinéarité)")


In [ ]:
# ============================================================
# CELLULE 9 : Scatterplots des variables les plus importantes
# ============================================================
top_vars = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF',
            'FullBath', 'YearBuilt', 'YearRemodAdd', 'GarageArea']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Relations entre Variables Clés et SalePrice', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, var in enumerate(top_vars):
    if var in train.columns:
        axes[i].scatter(train[var], train['SalePrice'],
                        alpha=0.3, color='steelblue', edgecolors='none', s=15)
        # Ligne de tendance
        z = np.polyfit(train[var].dropna(), train.loc[train[var].notna(), 'SalePrice'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(train[var].min(), train[var].max(), 100)
        axes[i].plot(x_line, p(x_line), 'r-', linewidth=2)
        
        corr = train[var].corr(train['SalePrice'])
        axes[i].set_title(f'{var}\n(r = {corr:.3f})', fontsize=10)
        axes[i].set_xlabel(var)
        axes[i].set_ylabel('SalePrice ($)')
        axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('../models/04_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Détection des Outliers

Les **outliers** (valeurs aberrantes) sont des observations très éloignées des autres.  
Ils peuvent fausser le modèle si on ne les traite pas.

Méthodes de détection :
- **IQR** : tout ce qui dépasse Q1 - 1.5×IQR ou Q3 + 1.5×IQR
- **Z-score** : tout ce qui est à plus de 3 écarts-types de la moyenne
- **Visualisation** : scatterplots et boxplots


In [ ]:
# ============================================================
# CELLULE 10 : Détection des outliers avec IQR
# ============================================================
# Variables numériques les plus importantes à vérifier
key_vars = ['GrLivArea', 'LotArea', 'TotalBsmtSF', '1stFlrSF',
            'GarageArea', 'MasVnrArea']

print("Détection des outliers (méthode IQR, facteur 3.0 pour cas extrêmes) :")
print("-" * 65)

outlier_summary = []
for var in key_vars:
    if var not in train.columns:
        continue
    Q1 = train[var].quantile(0.25)
    Q3 = train[var].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3.0 * IQR  # Facteur 3 pour extrêmes seulement
    upper = Q3 + 3.0 * IQR
    n_outliers = ((train[var] < lower) | (train[var] > upper)).sum()
    pct = n_outliers / len(train) * 100
    outlier_summary.append({'Variable': var, 'N_Outliers': n_outliers, 'Pct': pct})
    print(f"  {var:20s} : {n_outliers:4d} outliers extrêmes ({pct:.2f}%)")

print()
print("💡 On supprimera seulement les cas clairement aberrants documentés par Kaggle :")
print("   → GrLivArea > 4000 pieds² ET SalePrice < $300,000")


In [ ]:
# ============================================================
# CELLULE 11 : Visualisation des outliers - GrLivArea vs SalePrice
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Détection des Outliers : GrLivArea vs SalePrice', fontsize=13)

# Tous les points
axes[0].scatter(train['GrLivArea'], train['SalePrice'],
                alpha=0.4, color='steelblue', s=15, label='Normal')

# On identifie les outliers suspects
outliers_mask = (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
axes[0].scatter(train.loc[outliers_mask, 'GrLivArea'],
                train.loc[outliers_mask, 'SalePrice'],
                color='red', s=100, zorder=5, label=f'Outliers ({outliers_mask.sum()})')

axes[0].set_title('Avant traitement', fontsize=12)
axes[0].set_xlabel('Surface habitable (pieds²)')
axes[0].set_ylabel('Prix de vente ($)')
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Après suppression des outliers
train_clean = train[~outliers_mask]
axes[1].scatter(train_clean['GrLivArea'], train_clean['SalePrice'],
                alpha=0.4, color='green', s=15)
axes[1].set_title(f'Après suppression ({len(train) - len(train_clean)} cas retirés)', fontsize=12)
axes[1].set_xlabel('Surface habitable (pieds²)')
axes[1].set_ylabel('Prix de vente ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('../models/05_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Lignes d'entraînement après suppression outliers : {len(train_clean)}")
print(f"(Supprimées : {outliers_mask.sum()} — soit {outliers_mask.sum()/len(train)*100:.2f}%)")


## 4. Analyse des Valeurs Manquantes

Les valeurs manquantes (NaN = Not a Number) sont fréquentes dans les données réelles.  
Leur traitement doit être **justifié** et **cohérent** avec la signification des variables.


In [ ]:
# ============================================================
# CELLULE 12 : Analyse des valeurs manquantes
# ============================================================
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100)

missing_df = pd.DataFrame({
    'Nb_Manquants': missing,
    'Pourcentage': missing_pct
})

print(f"Colonnes avec valeurs manquantes : {len(missing_df)}")
print()
print(missing_df.to_string())


In [ ]:
# ============================================================
# CELLULE 13 : Visualisation des valeurs manquantes
# ============================================================
fig, ax = plt.subplots(figsize=(12, 8))

# Palette dégradée : rouge = beaucoup de NaN, vert = peu de NaN
colors = plt.cm.RdYlGn_r(missing_pct.values / 100)

bars = ax.barh(missing_df.index, missing_df['Pourcentage'],
               color=colors, edgecolor='black', linewidth=0.5)

# Ligne de référence à 50%
ax.axvline(x=50, color='red', linestyle='--', linewidth=2, label='50%')
ax.axvline(x=15, color='orange', linestyle='--', linewidth=2, label='15%')

ax.set_xlabel('% Valeurs Manquantes', fontsize=12)
ax.set_title('Valeurs Manquantes par Variable — Dataset Train', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# Affiche le pourcentage exact
for bar, val in zip(bars, missing_df['Pourcentage']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../models/06_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print("💡 STRATÉGIE D'IMPUTATION :")
print("   PoolQC, MiscFeature, Alley, Fence, FireplaceQu → NaN = Absent → 'None'")
print("   LotFrontage → médiane par quartier (Neighborhood)")
print("   Variables numériques de garage/sous-sol → 0 (absent = 0 m²)")
print("   Variables catégorielles restantes → mode (valeur la plus fréquente)")


## 5. Analyse des Variables Catégorielles Importantes

Les variables catégorielles contiennent des informations qualitatives (texte).  
On analyse en particulier le **Neighborhood** (quartier) et **OverallQual** (qualité).


In [ ]:
# ============================================================
# CELLULE 14 : Analyse du Neighborhood (quartier)
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(16, 12))
fig.suptitle('Impact du Quartier (Neighborhood) sur le Prix', fontsize=14, fontweight='bold')

# Médiane du prix par quartier, trié du plus cher au moins cher
neighborhood_price = train.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)

# Graphique du prix médian par quartier
axes[0].bar(range(len(neighborhood_price)), neighborhood_price.values,
            color=plt.cm.RdYlGn(np.linspace(0, 1, len(neighborhood_price))),
            edgecolor='black', linewidth=0.5)
axes[0].set_xticks(range(len(neighborhood_price)))
axes[0].set_xticklabels(neighborhood_price.index, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Prix Médian par Quartier', fontsize=12)
axes[0].set_ylabel('Prix Médian ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Boxplot par quartier (montre la dispersion aussi)
neighborhood_order = neighborhood_price.index.tolist()
data_by_neigh = [train[train['Neighborhood'] == n]['SalePrice'].values
                 for n in neighborhood_order]
bp = axes[1].boxplot(data_by_neigh, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], plt.cm.RdYlGn(np.linspace(0, 1, len(neighborhood_order)))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_xticks(range(1, len(neighborhood_order) + 1))
axes[1].set_xticklabels(neighborhood_order, rotation=45, ha='right', fontsize=9)
axes[1].set_title('Distribution des Prix par Quartier', fontsize=12)
axes[1].set_ylabel('Prix ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('../models/07_neighborhood_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Prix médian par quartier (Top 5 et Bottom 5) :")
print("
🏆 Quartiers les plus chers :")
print(neighborhood_price.head(5).apply(lambda x: f'${x:,.0f}'))
print("
📉 Quartiers les moins chers :")
print(neighborhood_price.tail(5).apply(lambda x: f'${x:,.0f}'))


In [ ]:
# ============================================================
# CELLULE 15 : Analyse de OverallQual (qualité générale)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Impact de la Qualité Générale (OverallQual) sur le Prix', fontsize=13)

# Prix moyen et médian par niveau de qualité
qual_stats = train.groupby('OverallQual')['SalePrice'].agg(['mean', 'median', 'count'])

# Barplot
x = qual_stats.index
axes[0].bar(x - 0.2, qual_stats['mean'], 0.4,
            label='Moyenne', color='#3498db', alpha=0.8)
axes[0].bar(x + 0.2, qual_stats['median'], 0.4,
            label='Médiane', color='#2ecc71', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xlabel('Qualité Générale (1=Très Mauvais, 10=Excellent)', fontsize=11)
axes[0].set_ylabel('Prix ($)')
axes[0].set_title('Prix Moyen/Médian par Qualité', fontsize=11)
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Distribution des qualités
axes[1].bar(qual_stats.index, qual_stats['count'],
            color=plt.cm.RdYlGn(qual_stats.index / 10), edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Qualité Générale', fontsize=11)
axes[1].set_ylabel('Nombre de maisons')
axes[1].set_title('Distribution des Niveaux de Qualité', fontsize=11)
# Affiche le nombre sur chaque barre
for i, (qual, count) in enumerate(qual_stats['count'].items()):
    axes[1].text(qual, count + 5, str(count), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../models/08_overall_quality.png', dpi=150, bbox_inches='tight')
plt.show()

print("Corrélation OverallQual / SalePrice :", round(train['OverallQual'].corr(train['SalePrice']), 4))


In [ ]:
# ============================================================
# CELLULE 16 : Relations non-linéaires - YearBuilt
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Relation entre Année de Construction et Prix', fontsize=13)

# Scatter YearBuilt vs SalePrice
axes[0].scatter(train['YearBuilt'], train['SalePrice'],
                alpha=0.3, color='steelblue', s=15)
axes[0].set_xlabel('Année de Construction')
axes[0].set_ylabel('Prix de Vente ($)')
axes[0].set_title('YearBuilt vs SalePrice', fontsize=11)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Prix médian par décennie
train['Decade'] = (train['YearBuilt'] // 10) * 10  # Ex: 1975 → 1970
decade_price = train.groupby('Decade')['SalePrice'].median()
axes[1].bar(decade_price.index, decade_price.values / 1000,
            width=8, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Décennie de Construction')
axes[1].set_ylabel('Prix Médian (K$)')
axes[1].set_title('Prix Médian par Décennie de Construction', fontsize=11)

plt.tight_layout()
plt.savefig('../models/09_yearbuilt_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Nettoyage
train = train.drop(columns=['Decade'])

print("Corrélation YearBuilt / SalePrice :", round(train['YearBuilt'].corr(train['SalePrice']), 4))
print("→ Relation non parfaitement linéaire mais tendance claire : maisons récentes = plus chères")


## Récapitulatif EDA — Conclusions

### 🔑 Principaux Enseignements

1. **Variable cible** : `SalePrice` est asymétrique → **transformation log(SalePrice)** nécessaire  
   - Asymétrie avant : ~1.88 → Asymétrie après log : ~0.12

2. **Variables les plus corrélées** (en valeur absolue) :
   - `OverallQual` (r=0.79) — Qualité générale
   - `GrLivArea` (r=0.71) — Surface habitable  
   - `GarageCars` (r=0.64) — Capacité du garage
   - `GarageArea` (r=0.62) — Surface du garage

3. **Outliers** : 4 maisons avec grande surface (>4000 pieds²) mais prix très bas → supprimées

4. **Valeurs manquantes** : 19 colonnes concernées  
   - La plupart signifient "absent" (PoolQC, GarageType...) → remplacer par 'None'
   - LotFrontage → médiane par quartier

5. **Variables catégorielles** : Le quartier a un impact **énorme** sur le prix  
   - NoRidge : médiane ~$335K vs MeadowV : médiane ~$85K

6. **Relations non-linéaires** : YearBuilt a une relation non-linéaire avec le prix


In [ ]:
# ============================================================
# CELLULE 17 : Résumé final des statistiques
# ============================================================
print("=" * 60)
print("   RÉSUMÉ EDA — DATASET HOUSE PRICES")
print("=" * 60)
print(f"   Lignes (train)          : {len(train)}")
print(f"   Colonnes totales        : {train.shape[1]}")
print(f"   Variables numériques    : {len(train.select_dtypes(include=[np.number]).columns)}")
print(f"   Variables catégorielles : {len(train.select_dtypes(include=['object']).columns)}")
print(f"   Valeurs manquantes      : {train.isnull().sum().sum()}")
print()
print(f"   SalePrice — Statistiques :")
print(f"     Minimum  : ${train['SalePrice'].min():,.0f}")
print(f"     Maximum  : ${train['SalePrice'].max():,.0f}")
print(f"     Moyenne  : ${train['SalePrice'].mean():,.0f}")
print(f"     Médiane  : ${train['SalePrice'].median():,.0f}")
print(f"     Asymétrie: {train['SalePrice'].skew():.4f}")
print()
print(f"   Top 5 corrélations avec SalePrice :")
top_corr = train.select_dtypes(include=[np.number]).corr()['SalePrice'].abs().sort_values(ascending=False)
for col, val in top_corr.iloc[1:6].items():
    print(f"     {col:20s} : {val:.4f}")
print("=" * 60)
